### RAG Pipeline - Data Ingestion to Vector DB Pipeline

In [12]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [14]:
# Read all the pdfs inside the directory
def process_all_pdfs(pdf_directory):
    all_documents=[]
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF Files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF Files to process

Processing: Capstone_Project_Proposal.pdf
Loaded 16 pages

Processing: CEM_Paper6.pdf
Loaded 6 pages

Total documents loaded: 22


In [15]:
all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-22T09:32:44+05:30', 'author': 'Anishka Chawla', 'moddate': '2026-03-22T09:32:44+05:30', 'source': '..\\data\\pdfs\\Capstone_Project_Proposal.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'Capstone_Project_Proposal.pdf', 'file_type': 'pdf'}, page_content='Multimodal EEG-EMG Integrated Assistive Robotic System \nCapstone Project Proposal \nSubmitted by: \n(102303901) Ishan Singh \n(102303879) Anishka Chawla  \n(102303635) Dharmpreet \n(102303646) Divit Dua \n(102323045) Garima Dahuja \n \nBE Third Year- COE \nCPG No. 121 \n \nUnder the Mentorship of \nDr. Garima Singh \nAssistant Professor \n \nComputer Science and Engineering Department \nThapar Institute of Engineering and Technology, Patiala \nMarch 2026'),
 Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-22T09:32:44+05:30', 'autho

In [16]:
# Text Splitting get into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [17]:
chunks=split_documents(all_pdf_documents)
chunks

Split 22 documents into 77 chunks

Example chunk:
Content: Multimodal EEG-EMG Integrated Assistive Robotic System 
Capstone Project Proposal 
Submitted by: 
(102303901) Ishan Singh 
(102303879) Anishka Chawla  
(102303635) Dharmpreet 
(102303646) Divit Dua 
(...
Metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-22T09:32:44+05:30', 'author': 'Anishka Chawla', 'moddate': '2026-03-22T09:32:44+05:30', 'source': '..\\data\\pdfs\\Capstone_Project_Proposal.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'Capstone_Project_Proposal.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-22T09:32:44+05:30', 'author': 'Anishka Chawla', 'moddate': '2026-03-22T09:32:44+05:30', 'source': '..\\data\\pdfs\\Capstone_Project_Proposal.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'Capstone_Project_Proposal.pdf', 'file_type': 'pdf'}, page_content='Multimodal EEG-EMG Integrated Assistive Robotic System \nCapstone Project Proposal \nSubmitted by: \n(102303901) Ishan Singh \n(102303879) Anishka Chawla  \n(102303635) Dharmpreet \n(102303646) Divit Dua \n(102323045) Garima Dahuja \n \nBE Third Year- COE \nCPG No. 121 \n \nUnder the Mentorship of \nDr. Garima Singh \nAssistant Professor \n \nComputer Science and Engineering Department \nThapar Institute of Engineering and Technology, Patiala \nMarch 2026'),
 Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-22T09:32:44+05:30', 'autho

### Embedding and VectorStoreDB

In [18]:

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\chawl\OneDrive\Documents\Programming\RAG\RAG1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [19]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


c:\Users\chawl\OneDrive\Documents\Programming\RAG\RAG1\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\chawl\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8858.72it/s]


Model loaded successfully. Embedding dimension: 384


### VectorStore

In [20]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [21]:
chunks

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-22T09:32:44+05:30', 'author': 'Anishka Chawla', 'moddate': '2026-03-22T09:32:44+05:30', 'source': '..\\data\\pdfs\\Capstone_Project_Proposal.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'source_file': 'Capstone_Project_Proposal.pdf', 'file_type': 'pdf'}, page_content='Multimodal EEG-EMG Integrated Assistive Robotic System \nCapstone Project Proposal \nSubmitted by: \n(102303901) Ishan Singh \n(102303879) Anishka Chawla  \n(102303635) Dharmpreet \n(102303646) Divit Dua \n(102323045) Garima Dahuja \n \nBE Third Year- COE \nCPG No. 121 \n \nUnder the Mentorship of \nDr. Garima Singh \nAssistant Professor \n \nComputer Science and Engineering Department \nThapar Institute of Engineering and Technology, Patiala \nMarch 2026'),
 Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-22T09:32:44+05:30', 'autho

In [22]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store in the vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 77 texts...


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.29it/s]

Generated embeddings with shape: (77, 384)
Adding 77 documents to vector store...
Successfully added 77 documents to vector store
Total documents in collection: 77


### Retriever Pipeline from VectorStore

In [23]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [24]:
rag_retriever

In [26]:
rag_retriever.retrieve("What is eeg")

Retrieving documents for query: 'What is eeg'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 74.66it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_f1773eef_24',
  'content': 'Page 12 of 16 \n \nenvironmental electromagnetic noise. Before digital processing can occur, the signals \nare routed through primary analog amplification and hardware-based filtering stages. \n● The EEG signal pathway utilizes precise bandpass filtering to isolate critical \nbrain rhythms, specifically isolating the Mu (8 –12 Hz) and Beta (13 –30 Hz) \nfrequency bands. These specific bands are highly correlated with the \nsensorimotor networks, effectively isolating the frequency range where event -\nrelated desynchronization occurs during motor imagery. \n● The EMG signal pathway employs a broader bandpass filter configuration \n(typically 20 –450 Hz) designed to capture the entire frequency spectrum of \nrapidly firing muscle fibers while aggressively attenuating low -frequency \nphysical motion artifacts and 50/60Hz mains interference. \n3. Real-Time Mathematical Feature Extraction  \nOnce conditioned, the analog voltages are digitized by th

In [28]:
rag_retriever.retrieve("prosthetic hand")

Retrieving documents for query: 'prosthetic hand'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 86.22it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_8b16494c_63',
  'content': 'As shown in Figure 4, the proposed design achieves a balanced\ncompromise between cost and functionality. The grip force of\napproximately3 Nper finger, while lower than commercial devices,\nis sufficient for many activities of daily living. The actuation speed\nof0.4 scompares favorably with both commercial and research\nprosthetics, addressing user concerns regarding responsiveness\n(Lake 2010).\nQuantitative performance evaluation focused on two main func-\ntional metrics: actuation speed and reliable gripping capability.\nThe wrist rotation mechanism, powered by a dedicated servo,\ncompletes a full 180°arc (from full pronation to full supination)\nin 0.45 seconds. This results in an average rotational speed of\n400°/s, which exceeds many commercial myoelectric prostheses\nand approaches the reflexive movement of a biological wrist. We\ncharacterized finger actuation performance through systematic\ngrasping tests with standardized objects, su

### Integration Vectordb Context pipeline With LLM output

In [36]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")

In [50]:
from langchain_groq import ChatGroq

In [42]:
### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
# groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [43]:
answer = rag_simple("What is eeg controlled robotic arm?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is eeg controlled robotic arm?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 65.22it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


A hybrid EEG-EMG based robotic arm control system that integrates brain signals (EEG) and muscle signals (EMG) to enable intent-driven robotic movement through a complete signal processing and control pipeline.


### Enhanced RAG Pipeline Features

In [47]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Prosthetic hand", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Prosthetic hand'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 125.41it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Prosthetic hand.
Sources: [{'source': 'CEM_Paper6.pdf', 'page': 3, 'score': 0.20411992073059082, 'preview': 'As shown in Figure 4, the proposed design achieves a balanced\ncompromise between cost and functionality. The grip force of\napproximately3 Nper finger, while lower than commercial devices,\nis sufficient for many activities of daily living. The actuation speed\nof0.4 scompares favorably with both comme...'}, {'source': 'CEM_Paper6.pdf', 'page': 0, 'score': 0.1925954818725586, 'preview': 'Design and Development of a Low-Cost\nEMG-Controlled Prosthetic Hand\nFakeraldeen Mohamed Abdalla Ali ID ∗,1, Youssouf Fadile Raye Bowou ID ∗,2, Ghassan Ali Mohammed Al-Shafali ID ∗,3 and Kamal\nAbdulrahman Adam Wady ID ∗,4\n∗Department of Electrical and Electronics Engineering, Faculty of Engineering a...'}, {'source': 'Capstone_Project_Proposal.pdf', 'page': 7, 'score': 0.1859450340270996, 'preview': 'Page 8 of 16 \n \nlightweight, human -like structure boasting six degrees of freedom

In [49]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("EEG Sensor", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'EEG Sensor'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 114.35it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
compact, wi reless, and edge -based EEG acquisition systems designed for practical 
deployment. Dou et al. (2026) developed a portable, wireless multi -channel EEG 
acquisition system optimized for real -time BCI applications. The system integrates 
a

nalog-to-digital conver sion (ADC), signal conditioning, power management, and 
microcontroller-based processing into a single wearable unit. Experimental evaluation 
demonstrated high signal quality, with input noise levels below 2.1 μV and a common-
mode rejection ratio (CMRR) exceeding 105 dB. The system was successfully validated 
through a steady -state visual evoked potential (SSVEP) based robotic feeding 
application, confirming its capability for real -time control. In a related study, 
Palanichamy et al. (2025) proposed a multimo dal approach combining EEG -based 
motor imagery with real-time gaze tracking to improve robotic control accuracy. Their

Page 6 of 16 
 
Literature Survey 
The development of a hybrid EEG -EMG based robotic arm control system requires 
integration of multiple domains, including bio -signal acquisition, signal processing, 
machine learning, and embedded control. Existing research highlights significant 
advancements in each of these areas while also e